# Replication dataset - Glasser360 cortical analyses


In [1]:
from itertools import combinations
from pathlib import Path
import sys

base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))
from utils.statistics import preprocess_df, safe_wilcoxon, friedman_test, pairwise_wilcoxon
import numpy as np
import pandas as pd
import pingouin as pg
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import multipletests
from scipy.stats import friedmanchisquare, wilcoxon
from scipy.stats import ttest_rel
import pickle
import os

# Anova

In [2]:
# parameters
tasks = ['Rest','0Back', '1Back', '2Back']
num_tasks = len(tasks)

df_path = f"{base_path}/RESULTS/replication_dataset/hurst_glasser360/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
atlas_path = f"{base_path}/DATA/templates/Glasser360_2mm.Centroid_RAS.csv"
output_dir =f'{base_path}/RESULTS/replication_dataset/glasser360/anova'
output_name = 'anova_results_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_rest_no_outliers.xlsx'

p_val_sig = 0.05 # significance threshold


In [3]:
# load data
data_df = pd.read_csv(df_path)
subject_ids = sorted(data_df["Subject"].unique())
parcel_ids = sorted(data_df["Parcel"].unique())

num_sub = len(subject_ids)
num_parcels = len(parcel_ids)

parcels = np.full(
    (num_sub, num_tasks, num_parcels),
    np.nan,
    dtype=float
)

for i, subj_id in enumerate(subject_ids):
    for task in range(num_tasks):
        for k_idx, parcel_id in enumerate(parcel_ids):

            sel = (
                (data_df["Subject"] == subj_id) &
                (data_df["Task"] == tasks[task]) &
                (data_df["Parcel"] == parcel_id)
            )

            vals = data_df.loc[sel, "Hursts"].values

            if vals.size == 0:
                continue
            elif vals.size == 1:
                parcels[i, task, k_idx] = vals[0]
            else:
                raise ValueError(
                    f"More than one value found for "
                    f"Subject={subj_id}, "
                    f"Task={tasks[task]}, "
                    f"Parcel={parcel_id}"
                )

# create mask to identify parcels with complete data across all tasks
mask_sub_parcel = ~np.isnan(parcels).any(axis=1)

# repeated measures ANOVA for each parcel
F_vec = np.full(num_parcels, np.nan)
p_vec = np.full(num_parcels, np.nan)
eta_p_vec = np.full(num_parcels, np.nan)
n_sub_vec = np.full(num_parcels, np.nan)

contrasts = list(combinations(tasks, 2))

task_to_idx = {t: i for i, t in enumerate(tasks)}
for a, b in contrasts:
    if a not in task_to_idx or b not in task_to_idx:
        raise ValueError(
            f"Task labels not found in `tasks`: missing {a} or {b}. Got tasks={tasks}"
        )

p_pair_raw = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
p_pair_fdr = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
t_pair = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}
n_pair = {
    f"{a}_vs_{b}": np.full(num_parcels, np.nan)
    for a, b in contrasts
}

# FIRST LOOP: calculate omnibus ANOVA for every parcel
for p in range(num_parcels):
    valid_sub = mask_sub_parcel[:, p]

    print(
        f"Analyzing parcel {p + 1}/{num_parcels} - "
        f"valid subjects: {valid_sub.sum()}/{num_sub}"
    )

    n_sub_p = int(valid_sub.sum())

    if n_sub_p < 2:
        continue

    data_p = parcels[valid_sub, :, p]
    n_sub_vec[p] = n_sub_p

    df_p = pd.DataFrame({
        "Subject": np.repeat(np.arange(n_sub_p), num_tasks),
        "Task": np.tile(tasks, n_sub_p),
        "Hurst": data_p.reshape(-1),
    })

    try:
        aov = AnovaRM(
            data=df_p,
            depvar="Hurst",
            subject="Subject",
            within=["Task"],
        ).fit()

        F_val = float(aov.anova_table.loc["Task", "F Value"])
        df_num = float(aov.anova_table.loc["Task", "Num DF"])
        df_den = float(aov.anova_table.loc["Task", "Den DF"])
        p_val = float(aov.anova_table.loc["Task", "Pr > F"])

    except Exception as e:
        print(f"ANOVA failed for parcel {p + 1}: {e}")
        continue

    F_vec[p] = F_val
    p_vec[p] = p_val

    eta_p_vec[p] = (
        (F_val * df_num) /
        (F_val * df_num + df_den)
    )

# FDR correction across omnibus ANOVAs
p_vec_fdr = np.full_like(p_vec, np.nan, dtype=float)
reject_fdr = np.zeros_like(p_vec, dtype=bool)

mask_ok = ~np.isnan(p_vec)

if mask_ok.any():
    reject_tmp, p_tmp, _, _ = multipletests(
        p_vec[mask_ok],
        alpha=p_val_sig,
        method="fdr_bh",
    )

    p_vec_fdr[mask_ok] = p_tmp
    reject_fdr[mask_ok] = reject_tmp

# SECOND LOOP: post-hoc only after significant FDR ANOVA
# Collect all post-hoc p-values across all significant parcels
all_posthoc_ps = []
all_posthoc_locations = []

for p in range(num_parcels):

    # Skip parcels whose omnibus ANOVA is not significant after FDR
    if not reject_fdr[p]:
        continue

    valid_sub = mask_sub_parcel[:, p]
    n_sub_p = int(valid_sub.sum())

    if n_sub_p < 2:
        continue

    data_p = parcels[valid_sub, :, p]

    for a, b in contrasts:
        key = f"{a}_vs_{b}"

        ia = task_to_idx[a]
        ib = task_to_idx[b]

        x = data_p[:, ia]
        y = data_p[:, ib]

        m = (~np.isnan(x)) & (~np.isnan(y))
        n = int(m.sum())

        n_pair[key][p] = n

        if n < 2:
            continue

        tstat, ppair = ttest_rel(
            x[m],
            y[m],
            nan_policy="omit",
        )

        t_pair[key][p] = float(tstat)
        p_pair_raw[key][p] = float(ppair)

        # Collect p-value together with its parcel and contrast
        if np.isfinite(ppair):
            all_posthoc_ps.append(float(ppair))
            all_posthoc_locations.append((key, p))

# One global FDR correction across all post-hoc comparisons
if len(all_posthoc_ps) > 0:
    _, all_posthoc_ps_fdr, _, _ = multipletests(
        all_posthoc_ps,
        alpha=p_val_sig,
        method="fdr_bh",
    )

    for (key, p), p_fdr in zip(
        all_posthoc_locations,
        all_posthoc_ps_fdr,
    ):
        p_pair_fdr[key][p] = float(p_fdr)


# add info and create dataframe
atlas_info = pd.read_csv(atlas_path)

f_values_df = pd.DataFrame({
    "Parcel": np.arange(1, num_parcels + 1),
    "F_value": F_vec,
    "p_value": p_vec,
    "p_value_fdr": p_vec_fdr,
    "significance": reject_fdr,
    "Partial_Eta_Squared": eta_p_vec,
    "n_subjects": n_sub_vec,
    "name": atlas_info["ROI Name"],
})

for key in p_pair_raw.keys():
    f_values_df[f"p_{key}"] = p_pair_raw[key]
    f_values_df[f"p_{key}_fdr"] = p_pair_fdr[key]
    f_values_df[f"t_{key}"] = t_pair[key]
    f_values_df[f"n_{key}"] = n_pair[key]

for key in p_pair_fdr.keys():
    f_values_df[f"sig_{key}_fdr"] = (
        f_values_df[f"p_{key}_fdr"] < p_val_sig
    )

    f_values_df.loc[
        f_values_df[f"p_{key}_fdr"].isna(),
        f"sig_{key}_fdr",
    ] = False

# Save results
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    output_name,
)

f_values_df.to_excel(
    output_path,
    index=False,
)

Analyzing parcel 1/360 - valid subjects: 33/34
Analyzing parcel 2/360 - valid subjects: 33/34
Analyzing parcel 3/360 - valid subjects: 32/34
Analyzing parcel 4/360 - valid subjects: 33/34
Analyzing parcel 5/360 - valid subjects: 33/34
Analyzing parcel 6/360 - valid subjects: 33/34
Analyzing parcel 7/360 - valid subjects: 31/34
Analyzing parcel 8/360 - valid subjects: 33/34
Analyzing parcel 9/360 - valid subjects: 30/34
Analyzing parcel 10/360 - valid subjects: 30/34
Analyzing parcel 11/360 - valid subjects: 31/34
Analyzing parcel 12/360 - valid subjects: 32/34
Analyzing parcel 13/360 - valid subjects: 33/34
Analyzing parcel 14/360 - valid subjects: 33/34
Analyzing parcel 15/360 - valid subjects: 31/34
Analyzing parcel 16/360 - valid subjects: 33/34
Analyzing parcel 17/360 - valid subjects: 32/34
Analyzing parcel 18/360 - valid subjects: 31/34
Analyzing parcel 19/360 - valid subjects: 32/34
Analyzing parcel 20/360 - valid subjects: 32/34
Analyzing parcel 21/360 - valid subjects: 33/34
A

In [4]:
print(np.sum(f_values_df['significance']), " significant parcels")

239  significant parcels


In [5]:
f_values_df.sort_values(by='Partial_Eta_Squared', ascending=False).head(3)

,Parcel,F_value,p_value,p_value_fdr,significance,Partial_Eta_Squared,n_subjects,name,p_Rest_vs_0Back,p_Rest_vs_0Back_fdr,...,p_1Back_vs_2Back,p_1Back_vs_2Back_fdr,t_1Back_vs_2Back,n_1Back_vs_2Back,sig_Rest_vs_0Back_fdr,sig_Rest_vs_1Back_fdr,sig_Rest_vs_2Back_fdr,sig_0Back_vs_1Back_fdr,sig_0Back_vs_2Back_fdr,sig_1Back_vs_2Back_fdr
157,158,35.627986,2.022713e-15,7.281768e-13,True,0.534730,32.0,L_V3CD,4.230474e-08,0.000012,...,0.081016,0.137860,1.803629,32.0,True,True,True,False,True,False
160,161,27.917771,7.466205e-13,1.343917e-10,True,0.482024,31.0,L_31pd,1.932024e-04,0.002348,...,0.000041,0.000786,4.803557,31.0,True,True,True,False,True,True
149,150,22.041770,1.273004e-10,9.652321e-09,True,0.440467,29.0,L_PGi,1.916265e-02,0.044901,...,0.000017,0.000404,5.184886,29.0,True,False,True,False,True,True


# 2-Way Anova

In [6]:
# parameters
tasks = ['Rest','0Back', '1Back', '2Back']
num_tasks = len(tasks)

df_path = f"{base_path}/RESULTS/replication_dataset/hurst_glasser360/dataframes/df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
atlas_path = f"{base_path}/DATA/templates/Glasser360_2mm.Centroid_RAS.csv"
output_dir =f'{base_path}/RESULTS/replication_dataset/glasser360/anova_2way'
output_name = 'anova_results_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_rest_no_outliers.xlsx'

p_val_sig = 0.05 # significance threshold

In [7]:
df = pd.read_csv(df_path)

network_order = np.unique(df["Network"].dropna()).tolist()

df = df.dropna(
    subset=["Hursts", "Subject", "Task", "Parcel", "Network"]
).copy()

df["Subject"] = df["Subject"].astype(str)

df["Task"] = pd.Categorical(
    df["Task"],
    categories=tasks,
    ordered=True
)

df["Network"] = pd.Categorical(
    df["Network"],
    categories=network_order,
    ordered=True
)

df_network = (
    df.groupby(
        ["Subject", "Task", "Network"],
        observed=True
    )
    .agg(
        Hursts_mean=("Hursts", "mean"),
        n_parcels=("Parcel", "nunique")
    )
    .reset_index()
)

print(df_network.head(7))

  Subject   Task Network  Hursts_mean  n_parcels
0       1   Rest       L     1.143634        174
1       1   Rest       R     1.126228        178
2       1  0Back       L     1.300886        180
3       1  0Back       R     1.285064        180
4       1  1Back       L     1.435796        178
5       1  1Back       R     1.389370        179
6       1  2Back       L     1.137332        179


In [8]:
anova_results = pg.rm_anova(
    data=df_network,
    dv="Hursts_mean",
    within=["Network", "Task"],
    subject="Subject",
    correction=True,
    detailed=True,
    effsize="np2"
)

display(anova_results)

os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, output_name)
f_values_df.to_excel(output_path, index=False)

,Source,SS,ddof1,ddof2,MS,F,p_unc,p_GG_corr,np2,eps,sphericity,W_spher,p_spher
0,Network,0.013374,1,32,0.013374,20.550560,0.000077,0.000077,0.391063,1.000000,True,NaN,1.000000
1,Task,0.341505,3,96,0.113835,4.175802,0.007969,0.013175,0.115431,0.814514,False,0.685810,0.041014
2,Network * Task,0.008901,3,96,0.002967,10.795705,0.000004,0.000026,0.252261,0.793608,False,0.693123,0.046557


# SVD

In [17]:
# Parameters
tasks = ['Rest', '0Back', '1Back', '2Back']
data_path = (
    f"{base_path}/RESULTS/replication_dataset/hurst_glasser360/dataframes/"
    "df_hurst_parcelwise_FIN_D4_modwt_1_5_padding_0_decimate_0_residuals_w_rest_IQR_cleaned.csv"
)

output_path = f"{base_path}/RESULTS/replication_dataset/glasser360/svd"

# SVD configurations
svd_configs = { 
    "all":  ['Rest', '0Back', '1Back', '2Back'],
    "tasks_only": ['0Back', '1Back', '2Back'],
}

# Preprocessing options
CENTER_COLUMNS = True
IMPUTE_MISSING = True

In [18]:
from utils.svd import (
    prepare_pivot_table,
    impute_missing_values,
    build_subject_task_matrix,
    run_svd_on_subject_task_matrix,
)

In [19]:
data_df = pd.read_csv(data_path)

os.makedirs(output_path, exist_ok=True)

all_summaries = []

for config_name, selected_tasks in svd_configs.items():

    print("\n" + "=" * 70)
    print(f"Running SVD configuration: {config_name}")
    print(f"Tasks included: {selected_tasks}")

    config_output_dir = f"{output_path}/{config_name}"

    summary = run_svd_on_subject_task_matrix(
        data_df=data_df,
        selected_tasks=selected_tasks,
        output_dir=config_output_dir,
        center_columns=CENTER_COLUMNS,
        impute_missing=IMPUTE_MISSING
    )

    all_summaries.append(summary)

    print(f"Saved results to: {config_output_dir}/svd_results.npz")
    print(f"Matrix shape: {summary['matrix_shape']}")
    print(f"Number of components: {summary['n_components']}")
    print(f"PC1 variance explained: {summary['variance_PC1'] * 100:.2f}%")

    if summary['n_components'] > 1:
        print(f"PC2 variance explained: {summary['variance_PC2'] * 100:.2f}%")

    print(
        f"Subjects with imputed values: "
        f"{summary['n_subjects_with_imputed_values']} / {summary['n_subjects']}"
    )

summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(f"{output_path}/svd_subject_task_summary.csv", index=False)

print("\n" + "=" * 70)
print("Done.")
print(f"Global summary saved to: {output_path}/svd_subject_task_summary.csv")


Running SVD configuration: all
Tasks included: ['Rest', '0Back', '1Back', '2Back']
(360, 136) (136,) (136, 136)
Saved results to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/glasser360/svd/all/svd_results.npz
Matrix shape: (360, 136)
Number of components: 136
PC1 variance explained: 46.75%
PC2 variance explained: 6.05%
Subjects with imputed values: 33 / 34

Running SVD configuration: tasks_only
Tasks included: ['0Back', '1Back', '2Back']
(360, 102) (102,) (102, 102)
Saved results to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/glasser360/svd/tasks_only/svd_results.npz
Matrix shape: (360, 102)
Number of components: 102
PC1 variance explained: 47.49%
PC2 variance explained: 5.20%
Subjects with imputed values: 31 / 34

Done.
Global summary saved to: /mnt/c/Users/matil/Desktop/Hurst_GitHub_refactored_complete/hurst-analysis/RESULTS/replication_dataset/glasser360/svd/svd_su

### Permutation test

In [20]:
path = f"{base_path}/RESULTS/replication_dataset/glasser360/svd/all/svd_results.npz"
n_perm = 2000
seed = 42

In [21]:
np.random.seed(seed)
data = np.load(path, allow_pickle=True)
X_centered = data['X_centered']
S = data['S']

n_components = min(X_centered.shape) 

var_explained_perm = np.zeros((n_perm, n_components))

for p in range(n_perm):
    X_perm = X_centered.copy()
    for j in range(X_centered.shape[1]):
        X_perm[:, j] = np.random.permutation(X_perm[:, j])

    _, S_perm, _ = np.linalg.svd(X_perm, full_matrices=False)
    var_explained_perm[p] = (S_perm**2) / (S_perm**2).sum()

var_explained = (S**2) / (S**2).sum()

q_pc1 = (var_explained_perm[:, 0] >= var_explained[0]).sum()
p_value_pc1 = (q_pc1 + 1) / (n_perm + 1)

q_pc2 = (var_explained_perm[:, 1] >= var_explained[1]).sum()
p_value_pc2 = (q_pc2 + 1) / (n_perm + 1)

print(f"PC1: var={var_explained[0]:.3f}, p={p_value_pc1:.4f}")
print(f"PC2: var={var_explained[1]:.3f}, p={p_value_pc2:.4f}")

PC1: var=0.468, p=0.0005
PC2: var=0.061, p=0.0005
